In [ ]:
# SIP Portfolio Dashboard (3 MFs) — interactive in VSCode Jupyter cell
# - Inputs: MF names, SIP/month, annual return %, years, start/end-of-month toggle
# - Outputs: 1 subplot figure (line + 3 donut pies) + portfolio summary
# - Formatting: ₹ symbol + Indian commas + compact ₹Cr/₹L

import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# -----------------------------
# Indian money formatting
# -----------------------------
RUPEE = "₹"

def indian_commas(num: float) -> str:
    n = int(round(num))
    s = str(abs(n))
    if len(s) <= 3:
        out = s
    else:
        last3 = s[-3:]
        rest = s[:-3]
        parts = []
        while len(rest) > 2:
            parts.append(rest[-2:])
            rest = rest[:-2]
        if rest:
            parts.append(rest)
        out = ",".join(reversed(parts)) + "," + last3
    return ("-" if n < 0 else "") + out

def fmt_rupee(num: float) -> str:
    return f"{RUPEE}{indian_commas(num)}"

def inr_compact(num: float) -> str:
    n = float(num)
    sign = "-" if n < 0 else ""
    n = abs(n)
    crore = 10_000_000
    lakh  = 100_000
    thousand = 1_000

    if n >= crore:
        return f"{sign}{RUPEE}{n/crore:.2f}Cr"
    if n >= lakh:
        return f"{sign}{RUPEE}{n/lakh:.2f}L"
    if n >= thousand:
        return f"{sign}{RUPEE}{n/thousand:.2f}K"
    return f"{sign}{RUPEE}{indian_commas(n)}"

# -----------------------------
# SIP simulation (month-by-month)
# -----------------------------
def monthly_rate_from_annual(R_percent: float) -> float:
    R = R_percent / 100.0
    return (1 + R) ** (1/12) - 1

def simulate_sip(P: float, R_percent: float, horizon_years: float, annuity_due: bool = True):
    i = monthly_rate_from_annual(R_percent)
    m = int(round(12 * horizon_years))
    balance = 0.0
    invested = 0.0
    invested_series, fv_series = [], []

    for month in range(1, m + 1):
        contrib = P
        invested += contrib

        # annuity_due=True => invest at start of month then earn return
        if annuity_due:
            balance = (balance + contrib) * (1 + i)
        else:
            balance = balance * (1 + i) + contrib

        invested_series.append(invested)
        fv_series.append(balance)

    invested_final = invested_series[-1] if invested_series else 0.0
    fv_final = fv_series[-1] if fv_series else 0.0
    gain_final = fv_final - invested_final

    return {
        "months": m,
        "monthly_rate": i,
        "invested_series": invested_series,
        "fv_series": fv_series,
        "invested_final": invested_final,
        "fv_final": fv_final,
        "gain_final": gain_final,
    }

# -----------------------------
# Widgets (inputs)
# -----------------------------
title = widgets.HTML("<h2>📊 SIP Portfolio Dashboard (3 Mutual Funds)</h2>")

years_w = widgets.IntSlider(value=10, min=1, max=40, step=1, description="Years", continuous_update=False)

timing_w = widgets.ToggleButtons(
    options=[("Start of Month", True), ("End of Month", False)],
    value=True,
    description="SIP timing",
)

mf1_name = widgets.Text(value="MF-1", description="MF-1 Name")
mf1_sip  = widgets.IntText(value=1500, description="MF-1 SIP")
mf1_r    = widgets.FloatText(value=12.0, description="MF-1 %")

mf2_name = widgets.Text(value="MF-2", description="MF-2 Name")
mf2_sip  = widgets.IntText(value=2000, description="MF-2 SIP")
mf2_r    = widgets.FloatText(value=11.0, description="MF-2 %")

mf3_name = widgets.Text(value="MF-3", description="MF-3 Name")
mf3_sip  = widgets.IntText(value=3000, description="MF-3 SIP")
mf3_r    = widgets.FloatText(value=13.0, description="MF-3 %")

run_btn = widgets.Button(description="Update Dashboard", button_style="success")

# Output areas
out_fig = widgets.Output()
out_summary = widgets.Output()

# -----------------------------
# Dashboard render function
# -----------------------------
def render_dashboard(_=None):
    # Collect inputs
    horizon_years = int(years_w.value)
    annuity_due = bool(timing_w.value)

    mfs = [
        {"name": mf1_name.value.strip() or "MF-1", "sip": float(mf1_sip.value), "r": float(mf1_r.value)},
        {"name": mf2_name.value.strip() or "MF-2", "sip": float(mf2_sip.value), "r": float(mf2_r.value)},
        {"name": mf3_name.value.strip() or "MF-3", "sip": float(mf3_sip.value), "r": float(mf3_r.value)},
    ]

    # Guardrails
    for mf in mfs:
        if mf["sip"] < 0:
            mf["sip"] = 0
        if mf["r"] < -100:
            mf["r"] = -100  # allow negative returns but keep sane

    # Simulate each MF
    results = []
    months_total = int(round(12 * horizon_years))
    for mf in mfs:
        res = simulate_sip(mf["sip"], mf["r"], horizon_years, annuity_due)
        res["name"] = mf["name"]
        res["sip"] = mf["sip"]
        res["r"] = mf["r"]
        results.append(res)

    # Portfolio series = sum of all MFs
    portfolio_invested = [0.0] * months_total
    portfolio_fv = [0.0] * months_total
    for r in results:
        for idx in range(months_total):
            portfolio_invested[idx] += r["invested_series"][idx]
            portfolio_fv[idx] += r["fv_series"][idx]

    portfolio_invested_final = portfolio_invested[-1]
    portfolio_fv_final = portfolio_fv[-1]
    portfolio_gain_final = portfolio_fv_final - portfolio_invested_final

    x_years = [(k + 1) / 12 for k in range(months_total)]

    names = [r["name"] for r in results]
    invested_vals = [r["invested_final"] for r in results]
    gain_vals = [r["gain_final"] for r in results]
    total_vals = [r["fv_final"] for r in results]

    # Build subplots
    fig = make_subplots(
        rows=2, cols=2,
        specs=[
            [{"type": "xy"}, {"type": "domain"}],
            [{"type": "domain"}, {"type": "domain"}],
        ],
        subplot_titles=(
            f"Portfolio Growth — Total: {fmt_rupee(portfolio_fv_final)} ({inr_compact(portfolio_fv_final)})",
            f"Invested Split — {fmt_rupee(portfolio_invested_final)}",
            f"Gains Split — {fmt_rupee(portfolio_gain_final)}",
            f"Total FV Split — {fmt_rupee(portfolio_fv_final)}",
        )
    )

    # Line chart: Total invested and total FV
    fig.add_trace(
        go.Scatter(
            x=x_years, y=portfolio_invested, mode="lines", name="Total Invested",
            customdata=[fmt_rupee(v) for v in portfolio_invested],
            hovertemplate="Years: %{x:.2f}<br>Total Invested: %{customdata}<extra></extra>",
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=x_years, y=portfolio_fv, mode="lines", name="Total Future Value",
            customdata=[fmt_rupee(v) for v in portfolio_fv],
            hovertemplate="Years: %{x:.2f}<br>Total Future Value: %{customdata}<extra></extra>",
        ),
        row=1, col=1
    )
    fig.update_xaxes(title_text="Years", row=1, col=1)
    fig.update_yaxes(title_text=f"Amount ({RUPEE})", row=1, col=1)

    # Donuts
    fig.add_trace(
        go.Pie(
            labels=names, values=invested_vals, hole=0.42,
            text=[f"{fmt_rupee(v)} ({inr_compact(v)})" for v in invested_vals],
            textinfo="label+percent",
            hovertemplate="%{label}<br>%{text}<extra></extra>",
            showlegend=False
        ),
        row=1, col=2
    )
    fig.add_trace(
        go.Pie(
            labels=names, values=gain_vals, hole=0.42,
            text=[f"{fmt_rupee(v)} ({inr_compact(v)})" for v in gain_vals],
            textinfo="label+percent",
            hovertemplate="%{label}<br>%{text}<extra></extra>",
            showlegend=False
        ),
        row=2, col=1
    )
    fig.add_trace(
        go.Pie(
            labels=names, values=total_vals, hole=0.42,
            text=[f"{fmt_rupee(v)} ({inr_compact(v)})" for v in total_vals],
            textinfo="label+percent",
            hovertemplate="%{label}<br>%{text}<extra></extra>",
            showlegend=False
        ),
        row=2, col=2
    )

    timing_label = "Start of Month" if annuity_due else "End of Month"
    fig.update_layout(
        title=f"3-MF SIP Portfolio Dashboard — {horizon_years} years — SIP timing: {timing_label}",
        height=900,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
        margin=dict(t=90, l=40, r=40, b=40),
    )

    # Display outputs
    with out_fig:
        clear_output(wait=True)
        display(HTML(fig.to_html(include_plotlyjs=True, full_html=False)))

    with out_summary:
        clear_output(wait=True)

        rows = ""
        for r in results:
            rows += f"""
            <tr>
              <td>{r['name']}</td>
              <td style="text-align:right;">{fmt_rupee(r['sip'])}</td>
              <td style="text-align:right;">{r['r']:.2f}%</td>
              <td style="text-align:right;">{fmt_rupee(r['invested_final'])} <span style="color:#666;">({inr_compact(r['invested_final'])})</span></td>
              <td style="text-align:right;">{fmt_rupee(r['gain_final'])} <span style="color:#666;">({inr_compact(r['gain_final'])})</span></td>
              <td style="text-align:right;"><b>{fmt_rupee(r['fv_final'])}</b> <span style="color:#666;">({inr_compact(r['fv_final'])})</span></td>
            </tr>
            """

        summary_html = f"""
        <h3>Portfolio Summary</h3>
        <ul>
          <li><b>Total Invested:</b> {fmt_rupee(portfolio_invested_final)} <span style="color:#666;">({inr_compact(portfolio_invested_final)})</span></li>
          <li><b>Total Gain:</b> {fmt_rupee(portfolio_gain_final)} <span style="color:#666;">({inr_compact(portfolio_gain_final)})</span></li>
          <li><b>Total Future Value:</b> <b>{fmt_rupee(portfolio_fv_final)}</b> <span style="color:#666;">({inr_compact(portfolio_fv_final)})</span></li>
        </ul>

        <h4>Breakdown by Mutual Fund</h4>
        <table style="border-collapse:collapse; width:100%;">
          <thead>
            <tr>
              <th style="border-bottom:1px solid #ddd; text-align:left; padding:6px;">MF</th>
              <th style="border-bottom:1px solid #ddd; text-align:right; padding:6px;">SIP / month</th>
              <th style="border-bottom:1px solid #ddd; text-align:right; padding:6px;">Return (p.a.)</th>
              <th style="border-bottom:1px solid #ddd; text-align:right; padding:6px;">Invested</th>
              <th style="border-bottom:1px solid #ddd; text-align:right; padding:6px;">Gain</th>
              <th style="border-bottom:1px solid #ddd; text-align:right; padding:6px;">Total FV</th>
            </tr>
          </thead>
          <tbody>{rows}</tbody>
        </table>
        """
        display(HTML(summary_html))

# Wire button + auto update on changes
run_btn.on_click(render_dashboard)
for w in [years_w, timing_w, mf1_name, mf1_sip, mf1_r, mf2_name, mf2_sip, mf2_r, mf3_name, mf3_sip, mf3_r]:
    w.observe(render_dashboard, names="value")

# Layout
mf1_box = widgets.VBox([mf1_name, mf1_sip, mf1_r])
mf2_box = widgets.VBox([mf2_name, mf2_sip, mf2_r])
mf3_box = widgets.VBox([mf3_name, mf3_sip, mf3_r])

controls = widgets.VBox([
    title,
    widgets.HBox([years_w, timing_w, run_btn]),
    widgets.HBox([mf1_box, mf2_box, mf3_box]),
])

display(controls)
display(out_fig)
display(out_summary)

# Initial render
render_dashboard()
